# Research 66 — ChimeraBoost and RealMLP-TD synthesis

Fixed, predeclared combinations only, no weight search: 80/20 reference + each candidate, and 80/10/10 reference + both. Compared with the reference and the four-seed VIP × region leader (0.752172 macro F1 / 0.765704 weighted F1). Uses saved OOF predictions only; Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
cols = [f"prob_{i}" for i in range(4)]


def load(name):
    frame = pd.read_csv(ROOT / "predictions" / name)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), name
    values = frame[cols].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), name
    return values / values.sum(axis=1, keepdims=True)


p_ref = load("segment-oof.csv")
p_cb = load("segment-chimeraboost-oof.csv")
p_rm = load("segment-realmlp-td-oof.csv")
candidates = {
    "Reference equal CatBoost/LightGBM": p_ref,
    "ChimeraBoost alone": p_cb,
    "RealMLP-TD alone": p_rm,
    "Fixed 80/20 reference + ChimeraBoost": 0.8 * p_ref + 0.2 * p_cb,
    "Fixed 80/20 reference + RealMLP-TD": 0.8 * p_ref + 0.2 * p_rm,
    "Fixed 80/10/10 reference + ChimeraBoost + RealMLP-TD": 0.8 * p_ref + 0.1 * p_cb + 0.1 * p_rm,
}
ref_folds = [f1_score(y[v], p_ref[v].argmax(1), average="macro") for _, v in folds]
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for k, (_, v) in enumerate(folds, start=1):
        row[f"fold_{k}_delta_vs_reference"] = (
            f1_score(y[v], pred[v], average="macro") - ref_folds[k - 1]
        )
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-chimeraboost-realmlp-synthesis.csv", index=False)
print("Leader for reference: four-seed VIP x region 80/20 = 0.752172 macro / 0.765704 weighted.")

,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_2_delta_vs_reference,fold_3_delta_vs_reference
0,Reference equal CatBoost/LightGBM,0.750895,0.764683,0.000000,0.000000,0.000000
1,ChimeraBoost alone,0.744862,0.758868,-0.007775,-0.005179,-0.005190
2,RealMLP-TD alone,0.754231,0.767629,0.003215,0.003765,0.003039
3,Fixed 80/20 reference + ChimeraBoost,0.751660,0.765359,-0.000139,0.001244,0.001178
4,Fixed 80/20 reference + RealMLP-TD,0.752264,0.765910,0.000901,0.001052,0.002162
5,Fixed 80/10/10 reference + ChimeraBoost + Real...,0.752064,0.765711,0.000775,0.001442,0.001287


Leader for reference: four-seed VIP x region 80/20 = 0.752172 macro / 0.765704 weighted.


## Research decision

Apply the predeclared rule to the 80/20 rows. Retain a candidate as a screening result only; no Test predictions are produced here.